# Optional Project - Colab Part2 Training

This notebook runs Part 2: Tiny LR sweep, five standard-parameterization model sizes, checkpoint sync to Google Drive, and scaling-law fit. Re-run interrupted training cells to resume from Drive checkpoints.

In [1]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
SWEEP_CONFIG = "configs/sweep_lr.yaml"
BEST_LR_JSON = "outputs/part2_lr_sweep_v2/best_lr.json"
DRIVE_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part2_lr_sweep_v2/best_lr.json"

In [2]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD

Cloning into '/content/optionalproject'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 151 (delta 85), reused 110 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 257.87 KiB | 835.00 KiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/optionalproject
Already on 'run'
Your branch is up to date with 'origin/run'.
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Already up to date.
run
583c1ea


In [3]:
# 2) Mount Google Drive for resumable checkpoints
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/svg-scaling/part2_v2
!mkdir -p /content/drive/MyDrive/svg-scaling/part2_lr_sweep_v2

Mounted at /content/drive


In [4]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:13 https:/

In [5]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN. Public dataset loading may still work.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [6]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('bf16_supported', torch.cuda.is_bf16_supported())

torch 2.10.0+cu128
cuda_available True
gpu NVIDIA A100-SXM4-80GB
bf16_supported True


In [7]:
# 6) Part 2 LR sweep on Tiny model
# If Colab disconnects, rerun this cell; each LR run resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_lr_sweep.py --config {SWEEP_CONFIG}

/content/optionalproject
[sweep] lr=0.001 run=tiny_lr_1.0e-03
/content/optionalproject/src/train/trainer.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.device.type == "cuda" and self.precision == "fp16")
README.md: 100% 654/654 [00:00<00:00, 3.77MB/s]
data/train-00000-of-00001.parquet: 100% 410M/410M [00:02<00:00, 156MB/s] 
data/validation-00000-of-00001.parquet: 100% 4.11M/4.11M [00:00<00:00, 10.0MB/s]
data/test-00000-of-00001.parquet: 100% 4.25M/4.25M [00:00<00:00, 10.3MB/s]
Generating train split: 100% 155571/155571 [00:01<00:00, 78431.39 examples/s] 
Generating validation split: 100% 1595/1595 [00:00<00:00, 97684.38 examples/s]
Generating test split: 100% 1590/1590 [00:00<00:00, 107793.10 examples/s]
{"type": "train", "step": 20, "tokens_seen": 140138, "next_row_index": 214, "train_loss": 8.15152928427309, "lr": 6.051873198847262e-05, "token

In [8]:
# 7) Inspect best LR
import json, os, shutil
from pathlib import Path
if not Path(BEST_LR_JSON).exists() and Path(DRIVE_BEST_LR_JSON).exists():
    Path(BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_BEST_LR_JSON, BEST_LR_JSON)
best = json.loads(Path(BEST_LR_JSON).read_text())
print(json.dumps(best, indent=2))
BEST_LR = best['learning_rate']
print('BEST_LR=', BEST_LR)

{
  "learning_rate": 0.004,
  "val_loss": 0.6363732951485246,
  "val_ppl": 1.8896153600304488,
  "val_tokens": 1006621.0,
  "run_name": "tiny_lr_4.0e-03",
  "global_step": 14612,
  "tokens_seen": 100470493,
  "num_parameters": 1579776,
  "num_parameters_non_embedding": 1055488,
  "wall_clock_seconds": 325.62759137153625,
  "tokens_per_second_epoch": 308544.1641379973,
  "peak_gpu_memory_gb": 1.3334121704101562
}
BEST_LR= 0.004


In [9]:
# 8) Train all five model sizes for exactly one epoch with the selected fixed LR
# If Colab disconnects, rerun this cell; each model resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_part2_all.py --best-lr-json {BEST_LR_JSON}

/content/optionalproject
[part2] /usr/bin/python3 scripts/run_train.py --config configs/train_tiny.yaml --learning-rate 0.004
/content/optionalproject/src/train/trainer.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.device.type == "cuda" and self.precision == "fp16")
{"type": "train", "step": 20, "tokens_seen": 140138, "next_row_index": 214, "train_loss": 7.825085394736588, "lr": 0.00024207492795389049, "tokens_per_second": 152810.19892265135, "gpu_memory_gb": 1.2935857772827148, "elapsed_seconds": 0.9180853366851807}
{"type": "train", "step": 40, "tokens_seen": 279764, "next_row_index": 435, "train_loss": 6.531014031887007, "lr": 0.0004726224783861672, "tokens_per_second": 341383.92994572915, "gpu_memory_gb": 1.3225154876708984, "elapsed_seconds": 1.3276193141937256}
{"type": "train", "step": 60, "tokens_seen": 417436, "next_row_index": 630, "t

In [10]:
# 9) Fit Part 2 scaling law and create plot/table
%cd $REPO_DIR
!python scripts/fit_scaling_law.py --runs-dir outputs/part2_v2 --output-dir outputs/part2_analysis_v2
!mkdir -p /content/drive/MyDrive/svg-scaling/part2_analysis_v2
!cp -r outputs/part2_analysis_v2/* /content/drive/MyDrive/svg-scaling/part2_analysis_v2/

/content/optionalproject
{
  "alpha": 1.6154817978376846e-16,
  "a": 0.4745636624228082,
  "c": 0.5872712142702109,
  "table": "outputs/part2_analysis_v2/part2_scaling_table.csv"
}


In [11]:
# 10) Inspect key Part 2 outputs
from pathlib import Path
import json
for p in sorted(Path('outputs/part2_v2').glob('*/final_metrics.json')):
    m=json.loads(p.read_text())
    print(p.parent.name, {k:m.get(k) for k in ['num_parameters','val_loss','val_ppl','tokens_seen','wall_clock_seconds','peak_gpu_memory_gb']})
fit_path=Path('outputs/part2_analysis_v2/part2_scaling_fit.json')
if fit_path.exists():
    print(json.dumps(json.loads(fit_path.read_text())['fit'], indent=2))

large {'num_parameters': 34670592, 'val_loss': 1.596881548320236, 'val_ppl': 4.937610690574898, 'tokens_seen': 100470493, 'wall_clock_seconds': 1436.2385802268982, 'peak_gpu_memory_gb': 4.720276832580566}
medium {'num_parameters': 13006848, 'val_loss': 0.8203044681181949, 'val_ppl': 2.2711912375945977, 'tokens_seen': 100470493, 'wall_clock_seconds': 707.1197338104248, 'peak_gpu_memory_gb': 2.6203012466430664}
small {'num_parameters': 3849216, 'val_loss': 0.5873720299113822, 'val_ppl': 1.7992538117252483, 'tokens_seen': 100470493, 'wall_clock_seconds': 517.9249505996704, 'peak_gpu_memory_gb': 1.845506191253662}
tiny {'num_parameters': 1579776, 'val_loss': 0.6318138574932896, 'val_ppl': 1.8810193878949564, 'tokens_seen': 100470493, 'wall_clock_seconds': 325.2511975765228, 'peak_gpu_memory_gb': 1.3334121704101562}
xl {'num_parameters': 89774592, 'val_loss': 1.6728024796219862, 'val_ppl': 5.327075919278298, 'tokens_seen': 100470493, 'wall_clock_seconds': 2770.1954731941223, 'peak_gpu_memor